
<div class="alert alert-block alert-info">  
    
## Import & Connections
</div>

In [2]:
import numpy as np
import pandas as pd
import hashlib
import glob
import time
import os
import re
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 50)

In [4]:
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [14]:
start_date = '20250115'
end_date = '20250122'
response_type = 'ONLINE_SALES'
pagename = 'PostOrderScreen'
campaign_name = 'Exp1_HeroV2_TataNeu'
format = 'heroBannerV2'

In [11]:
print(os.getcwd())
local_dataset = '/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/hero-banner'
print(local_dataset)

/Users/E2074/analytics/customer/Ads-Monetisation/page-format-experiment
/Users/E2074/local-datasets/customer/ads-monetisation/page-format-experiment/hero-banner


<div class="alert alert-block alert-warning">  
    
## Datasets & Functions
</div>

In [12]:
start = datetime.strptime(start_date, '%Y%m%d')
end = datetime.strptime(end_date, '%Y%m%d')

# Generate the list of dates
date_range = [(start + timedelta(days=i)).strftime('%Y%m%d') for i in range((end - start).days + 1)]
print(date_range)

['20250115', '20250116', '20250117', '20250118', '20250119', '20250120', '20250121', '20250122']


In [17]:
def get_customer_metadata(date_range):

    all_df = []
    
    for date in date_range:
    
        customer_metadata = f"""
        
            with response as (
    
                select
                    distinct
                    yyyymmdd,
                    city,
                    os,
                    json_extract_scalar(metadata1, '$.campaignName') AS campaignName,
                    json_extract_scalar(metadata1, '$.slotUnitId') AS slotUnitId,
                    userid,
                    service,
                    orderstatus,
                    -- totalcardcount,
                    -- cardposition,
                    coalesce(aduuid, adid) response_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Response'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, 'Exp1_HeroV2_TataNeu')
                ),
                
                impression as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    coalesce(aduuid, adid) impression_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Impression'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, 'Exp1_HeroV2_TataNeu')
                ),
                
                viewable_impression as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    coalesce(aduuid, adid) viewable_impression_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Viewable_Impression'
                    and pagename = '{pagename}'
                    and format = '{format}'
                    and regexp_like(metadata1, 'Exp1_HeroV2_TataNeu')
                ),
                
                click as (
                
                select
                    distinct
                    yyyymmdd,
                    userid,
                    coalesce(aduuid, adid) click_id
                from 
                    canonical.iceberg_log_telemetry_ads_impressions_immutable
                where 
                    yyyymmdd = '{date}'
                    and os = 'android'
                    and responsetype = '{response_type}'
                    and eventname = 'Ad_Click'
                    and pagename = '{pagename}'
                    and format = '{format}'
                )
                
                select
                    a.*,
                    impression_id,
                    coalesce(click_id, viewable_impression_id) viewable_impression_id,
                    click_id
                from 
                    response a
                left join 
                    impression b
                    on a.yyyymmdd = b.yyyymmdd
                    and a.userid = b.userid
                    and a.response_id = b.impression_id
                left join 
                    viewable_impression c
                    on b.yyyymmdd = c.yyyymmdd
                    and b.userid = c.userid
                    and b.impression_id = c.viewable_impression_id
                left join 
                    click d
                    on b.yyyymmdd = d.yyyymmdd
                    and b.userid = d.userid
                    and b.impression_id = d.click_id
        """
        
        df = pd.read_sql(customer_metadata, connection)
        all_df.append(df)
        df.to_parquet(local_dataset + '/data_dump/Exp1_HeroV2_TataNeu_{}.parquet'.format(date), index=False)
    
    # Concatenate all DataFrames
    final_df = pd.concat(all_df, ignore_index=True)

    return final_df


# df_data = get_customer_metadata(date_range)

In [18]:
def get_customer_segment():    
    
    customer_segment = f"""
    
        with response as (

            select
                distinct
                userid
            from 
                canonical.iceberg_log_telemetry_ads_impressions_immutable
            where 
                yyyymmdd >= '{start_date}'
                and yyyymmdd <= '{end_date}'
                and os = 'android'
                and responsetype = '{response_type}'
                and eventname = 'Ad_Response'
                and pagename = '{pagename}'
                and format = '{format}'
                and regexp_like(metadata1, 'Exp1_HeroV2_TataNeu')
            ),

            segment as (
    
            select
                customer_id,
                gender_tag,
                taxi_ltr_segment,
                taxi_retention_segments,
                taxi_regularity_segment,
                taxi_fe_regularity_segment,
                taxi_income_segment,
                taxi_service_affinity,
                taxi_gc_tag
            from 
                datasets.iallocator_customer_segments
            where 
                run_date = '2025-01-14'
            )
            
            select
                userid,
                coalesce(gender_tag, 'UNKNOWN') gender,
                coalesce(taxi_ltr_segment, 'NEW') ltr_segment,
                coalesce(taxi_retention_segments, 'UNKNOWN') retention_segments,
                coalesce(taxi_regularity_segment, 'UNKNOWN') regularity_segment,
                coalesce(taxi_fe_regularity_segment, 'UNKNOWN') fe_regularity_segment,
                coalesce(taxi_income_segment, 'UNKNOWN') income_segment,
                coalesce(taxi_service_affinity, 'UNKNOWN') service_affinity,
                coalesce(taxi_gc_tag, 'UNKNOWN') gc_tag
            from 
                response
            left join 
                segment
                on userid = customer_id
            
    """
    
    df = pd.read_sql(customer_segment, connection)
    df.to_parquet(local_dataset + '/customer_segment.parquet', index=False)

    return df

# df_data = get_customer_segment()


<div class="alert alert-block alert-info">  
    
## Load data
</div>

In [19]:
def get_customer_metadata():
    
    # Use glob to find all Parquet files in the directory
    parquet_files = glob.glob(os.path.join(local_dataset + '/data_dump/', "*.parquet"))
    customer_metadata = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

    customer_segment = pd.read_parquet(local_dataset + '/customer_segment.parquet')
    
    return customer_metadata, customer_segment

df_cux_metadata, df_cux_segment = get_customer_metadata()

In [20]:
df_cux_metadata.shape, df_cux_segment.shape

((1537000, 12), (557882, 9))

In [21]:
df_cux_metadata.head(1)

,yyyymmdd,city,os,campaignName,slotUnitId,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id
0,20250119,Hyderabad,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,676f87131d37e3d34876a364,Auto,started,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,None


In [22]:
df_cux_segment.head(1)

,userid,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag
0,614fb522f2602cb2282b4e90,MALE,PHH,DORMANT,BI_WEEKLY,BI_WEEKLY,LOW_INCOME,UNKNOWN,UNKNOWN


In [23]:
df = pd.merge(df_cux_metadata, df_cux_segment, how='left', on = 'userid')
df.head(1).T

,0
yyyymmdd,20250119
city,Hyderabad
os,android
campaignName,Exp1_HeroV2_TataNeu
slotUnitId,PostOrderStarted2
userid,676f87131d37e3d34876a364
service,Auto
orderstatus,started
response_id,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...
impression_id,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...


<div class="alert alert-block alert-warning">  
    
## Funnel Func
</div>

In [70]:

def get_funnel(df, groupby):

    df_temp = df\
                .groupby(groupby)\
                .agg(responses = ('response_id', 'nunique'),
                     impression = ('impression_id', 'nunique'),
                     viewable_impression = ('viewable_impression_id', 'nunique'),
                     click = ('click_id', 'nunique'),
                    )\
                .reset_index()

    df_temp['% render rate'] = (df_temp['impression']*100.00/df_temp['responses']).round(2)
    df_temp['% view'] = (df_temp['viewable_impression']*100.00/df_temp['impression']).round(2)
    df_temp['% ctr'] = (df_temp['click']*100.00/df_temp['viewable_impression']).round(2)

    df_temp['format'] = format
    
    columns = ['format'] + groupby + ['responses', '% render rate', '% view', '% ctr']
    
    return df_temp[columns].sort_values(by='% ctr', ascending=False)

<div class="alert alert-block alert-success">  
    
## Analysis
</div>

In [82]:
df.head(5)

,yyyymmdd,city,os,campaignName,slotUnitId,userid,service,orderstatus,response_id,impression_id,viewable_impression_id,click_id,gender,ltr_segment,retention_segments,regularity_segment,fe_regularity_segment,income_segment,service_affinity,gc_tag
0,20250119,Hyderabad,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,676f87131d37e3d34876a364,Auto,started,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,6097aab4-4c2c-4293-806c-82b6db31000f676f87131d...,None,MALE,HH,HH,UNKNOWN,MONTHLY,LOW_INCOME,UNKNOWN,UNKNOWN
1,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,66a5dff7fc10f2126b031cf1,Auto,onTheWay,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,f753d406-7a63-4071-889d-c80d4ce3b72166a5dff7fc...,None,FEMALE,PHH,GOLD,BI_WEEKLY,BI_WEEKLY,MEDIUM_INCOME,UNKNOWN,UNKNOWN
2,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderOnTheWay2,672d535d323c904f9840cfbe,Link,onTheWay,5973be36-c638-412f-ac05-b6515030844f672d535d32...,5973be36-c638-412f-ac05-b6515030844f672d535d32...,5973be36-c638-412f-ac05-b6515030844f672d535d32...,None,MALE,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,MEDIUM_INCOME,UNKNOWN,UNKNOWN
3,20250119,Hyderabad,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,6208fc428e29c807291bdaf0,Link,started,c7c2e995-5399-426f-a9b6-3035b605d3026208fc428e...,c7c2e995-5399-426f-a9b6-3035b605d3026208fc428e...,c7c2e995-5399-426f-a9b6-3035b605d3026208fc428e...,None,MALE,PHH,SILVER,MONTHLY,MONTHLY,MEDIUM_INCOME,UNKNOWN,UNKNOWN
4,20250119,Bangalore,android,Exp1_HeroV2_TataNeu,PostOrderStarted2,64e5b8af7284de2fb4552e7d,Auto,started,ff12a600-4f51-4499-b9d9-0546f142aecb64e5b8af72...,ff12a600-4f51-4499-b9d9-0546f142aecb64e5b8af72...,None,None,MALE,PHH,DORMANT,RARE_NEED,UNKNOWN,HIGH_INCOME,UNKNOWN,UNKNOWN


#### Bivariate data 

In [83]:
display(get_funnel(df, ['os']))

,format,os,responses,% render rate,% view,% ctr
0,heroBannerV2,android,1537000,80.37,74.23,0.28


In [71]:
display(get_funnel(df, ['yyyymmdd']))

,format,yyyymmdd,responses,% render rate,% view,% ctr
2,heroBannerV2,20250119,308269,80.84,73.43,0.31
3,heroBannerV2,20250120,309985,80.47,74.42,0.28
4,heroBannerV2,20250121,308642,79.93,74.12,0.28
1,heroBannerV2,20250118,306764,80.66,74.62,0.27
5,heroBannerV2,20250122,303250,79.93,74.58,0.27
0,heroBannerV2,20250117,90,90.00,60.49,0.00


In [72]:
display(get_funnel(df, ['slotUnitId']))

,format,slotUnitId,responses,% render rate,% view,% ctr
2,heroBannerV2,PostOrderStarted2,377491,78.93,79.56,0.51
1,heroBannerV2,PostOrderOnTheWay2,814829,81.43,68.09,0.22
0,heroBannerV2,PostOrderArrived2,344680,79.44,83.31,0.15


In [73]:
display(get_funnel(df, ['service']))

,format,service,responses,% render rate,% view,% ctr
11,heroBannerV2,CabPremium,8232,78.04,80.70,1.00
2,heroBannerV2,Auto NCR,1450,79.93,74.37,0.58
10,heroBannerV2,CabEconomy,398889,79.55,72.86,0.31
9,heroBannerV2,C2C,4097,78.62,72.18,0.30
6,heroBannerV2,Bike Lite,40495,81.83,74.46,0.29
0,heroBannerV2,Auto,473986,80.05,74.25,0.27
13,heroBannerV2,Link,589296,81.12,74.97,0.26
7,heroBannerV2,Bike Metro,18804,80.39,76.81,0.18
1,heroBannerV2,Auto C2C,173,73.99,66.41,0.00
3,heroBannerV2,Auto Pet,361,76.73,45.49,0.00


In [74]:
display(get_funnel(df, ['gender']))

,format,gender,responses,% render rate,% view,% ctr
3,heroBannerV2,UNKNOWN,135736,82.42,72.58,0.41
2,heroBannerV2,OTHERS,6246,83.75,68.76,0.39
0,heroBannerV2,FEMALE,307611,80.93,74.75,0.28
1,heroBannerV2,MALE,1087407,79.93,74.32,0.26


In [75]:
display(get_funnel(df, ['ltr_segment']))

,format,ltr_segment,responses,% render rate,% view,% ctr
1,heroBannerV2,NEW,106177,83.03,71.99,0.45
3,heroBannerV2,UNKNOWN,26853,83.07,66.38,0.40
0,heroBannerV2,HH,181661,81.39,72.84,0.36
2,heroBannerV2,PHH,1222309,79.92,74.82,0.25


In [76]:
display(get_funnel(df, ['retention_segments']))

,format,retention_segments,responses,% render rate,% view,% ctr
8,heroBannerV2,UNKNOWN,133030,83.03,70.85,0.44
4,heroBannerV2,INACTIVE,117273,80.98,73.97,0.37
3,heroBannerV2,HH,74987,81.15,72.64,0.33
0,heroBannerV2,DORMANT,251107,79.99,74.31,0.30
7,heroBannerV2,SILVER,119183,79.68,74.89,0.29
2,heroBannerV2,GOLD,222855,79.86,74.72,0.27
6,heroBannerV2,PRIME,11408,79.69,75.43,0.25
5,heroBannerV2,PLATINUM,275544,79.75,74.93,0.23
1,heroBannerV2,ELITE,331613,80.31,74.83,0.21


In [77]:
display(get_funnel(df, ['regularity_segment']))

,format,regularity_segment,responses,% render rate,% view,% ctr
4,heroBannerV2,UNKNOWN,314691,82.08,71.99,0.39
3,heroBannerV2,RARE_NEED,165352,80.01,74.82,0.32
2,heroBannerV2,MONTHLY,156668,80.01,75.15,0.28
0,heroBannerV2,BI_WEEKLY,284728,79.77,74.92,0.26
5,heroBannerV2,WEEKLY,343799,79.61,75.02,0.24
1,heroBannerV2,DAILY,271762,80.37,74.27,0.20


In [78]:
display(get_funnel(df, ['fe_regularity_segment']))

,format,fe_regularity_segment,responses,% render rate,% view,% ctr
4,heroBannerV2,UNKNOWN,340366,81.40,73.50,0.38
3,heroBannerV2,RARE_NEED,82809,80.82,73.24,0.32
2,heroBannerV2,MONTHLY,222492,80.20,74.31,0.30
0,heroBannerV2,BI_WEEKLY,365754,79.85,74.60,0.26
5,heroBannerV2,WEEKLY,226766,79.72,75.06,0.23
1,heroBannerV2,DAILY,298813,80.31,74.20,0.20


In [79]:
display(get_funnel(df, ['income_segment']))

,format,income_segment,responses,% render rate,% view,% ctr
3,heroBannerV2,UNKNOWN,399929,79.98,73.23,0.35
2,heroBannerV2,MEDIUM_INCOME,481159,85.19,75.85,0.27
1,heroBannerV2,LOW_INCOME,81433,86.61,75.79,0.26
0,heroBannerV2,HIGH_INCOME,574479,75.71,73.19,0.25


In [80]:
display(get_funnel(df, ['service_affinity']))

,format,service_affinity,responses,% render rate,% view,% ctr
7,heroBannerV2,UNKNOWN,831450,80.58,73.98,0.33
1,heroBannerV2,LINK_AUTO,40703,80.82,75.79,0.28
4,heroBannerV2,ONLY_AUTO,187133,79.36,74.72,0.26
0,heroBannerV2,AUTO_CAB,17062,77.46,73.68,0.22
2,heroBannerV2,LINK_CAB,10504,81.46,72.34,0.21
3,heroBannerV2,NO_AFFINITY,93241,79.45,73.39,0.21
5,heroBannerV2,ONLY_CAB,50931,78.16,72.19,0.21
6,heroBannerV2,ONLY_LINK,305976,81.12,75.06,0.19


In [81]:
display(get_funnel(df, ['gc_tag']))

,format,gc_tag,responses,% render rate,% view,% ctr
2,heroBannerV2,UNKNOWN,1226827,80.66,74.09,0.29
1,heroBannerV2,NON_GC,285660,79.17,74.69,0.23
0,heroBannerV2,GC,24513,79.82,75.82,0.18
